In [1]:
import torch
from transformers import RobertaTokenizer,RobertaModel,RobertaForMaskedLM, pipeline, BertForMaskedLM, BertTokenizer, BertModel, AlbertConfig, AlbertModel, AlbertTokenizer, AlbertForMaskedLM , TFAutoModel, AutoModelForMaskedLM, AutoTokenizer, AutoModelForCausalLM
import math
import numpy as np
import json
import random
import pandas as pd
import tensorflow as tf
from torch.nn import CrossEntropyLoss
from tqdm.notebook import tqdm
import re
from scipy.optimize import linear_sum_assignment

In [2]:
import sys
sys.path.append('..')
import SMI_Methods

In [3]:
import logging
import os
import sys
logging.basicConfig(level=logging.INFO)
import warnings
warnings.filterwarnings('ignore')

In [7]:
# Dictionary to map Czech characters to their Polish equivalents.
czech = {
    'č': 'cz', 
    'š': 'sz', 
    'ř': 'rz', 
    'ž': 'ż', 
    'ý': 'y', 
    'á': 'a', 
    'í': 'i', 
    'é': 'e', 
    'ě': 'e', 
    'ú': 'u', 
    'ů': 'u',
    'v': 'w'
}

slovak = {
    'á': 'o',
    'ä': 'e',
    'č': 'cz',
    'ď': 'd',
    'dz': 'dż',
    'dž': 'dż',
    'é': 'e',
    'í': 'i',
    'ľ': 'l',
    'ĺ': 'l',
    'ň': 'ń',
    'ó': 'o',
    'ô': 'u',
    'ŕ': 'r',
    'š': 'sz',
    'ť': 't',
    'ú': 'u',
    'ý': 'i',
    'ž': 'ż',
}
croatian = {
    'č': 'cz',
    'đ': 'dż',
    'dž': 'dż',
    'š': 'sz',
    'ž': 'ż',
}
slovene = {
    'č': 'cz',
    'š': 'sz',
    'ž': 'ż',
    'đ': 'dż',
}

def getKey(lang):
    d = {
        "czech":czech,
        "slovak":slovak,
        "croatian":croatian,
        "slovene":slovene
    }
    return d[lang]

def transliterate(text, lang):
    """Transcribes a text into Polish phonetically."""  
    #determine key to use
    key = getKey(lang)    
    # We need to replace longer substrings first to avoid partial matches.
    for char, polish_char in sorted(key.items(), key=lambda x: -len(x[0])):
        text = text.replace(char, polish_char)
        text = text.replace(char.upper(), polish_char.upper())  # Replace uppercase characters.
    return text

def replace_in_nested_list(nested_list,lang):
    """
    Perform the replacement operation on the deepest sublists of arbitrary depth.
    :param nested_list: The nested list to operate on
    :return: The nested list with replaced strings.
    """
    if not isinstance(nested_list, list):
        return transliterate(nested_list, lang)
    return [replace_in_nested_list(sublist, lang) for sublist in nested_list]


In [32]:
def count_tokens(model, tokenizer, data):
    r = 0
    for d in data:
        for s in d:
            for l in s:
                r += len(tokenizer.tokenize(l))
    return r

def see_tokens(model, tokenizer, data):
    r = 0
    for d in data:
        for s in d:
            for l in s:
                print(tokenizer.tokenize(l), len(tokenizer.tokenize(l)))
                break

In [61]:
langs = ["Czech","Slovak","Croatian","Slovene"]
def token_tester(lang):
    print(lang)
    word_list = SMI_Methods.prep_words(lang)
    data = SMI_Methods.prep_data(lang, word_list)
    #print(data[0][0])
    print(count_tokens(model, tokenizer, data))
    see_tokens(model, tokenizer, data)
    print("-------------------------------------------------------------------------------")
    data = replace_in_nested_list(data, lang.lower())
    #print(data[0][0])
    print(count_tokens(model, tokenizer, data))
    see_tokens(model, tokenizer, data)
    print("-------------------------------------------------------------------------------")
    print("-------------------------------------------------------------------------------")
    
def run_tests(model, tokenizer, lang, BERT = False):
    word_list = SMI_Methods.prep_words(lang)
    data = SMI_Methods.prep_data(lang, word_list)
    
       
    word_list = replace_in_nested_list(word_list, lang.lower())
    data = replace_in_nested_list(data, lang.lower())
    
    token_count = count_tokens(model, tokenizer, data)
    
    results = []
    for desc, d in zip(["1 Line Data", "3 Line Data", "3 Line Data (unfilled)"], [data[0], data[1], data[2]]):
        scores_line = [0,0,0,0]
        for i in range(4):
            sc = (SMI_Methods.score_model_bert(model, tokenizer, d[i], word_list[i]))if BERT else (SMI_Methods.score_model(model, tokenizer, d[i], word_list[i]))
            scores_line[0] += sc[0]
            scores_line[1] += sc[1]
            scores_line[2] += sc[2]
            scores_line[3] += sc[3]
        results.append({
            "Language": lang,
            "Description": desc,
            "Token Count": token_count,
            "Top 1 Score": scores_line[0]/4,
            "Top 3 Score": scores_line[1]/4,
            "COS": scores_line[2]/4,
            "GAS": scores_line[3]/4
        })

    df = pd.DataFrame(results)
    return df


In [69]:
tokenizer = AutoTokenizer.from_pretrained("sdadas/polish-gpt2-small")
model = AutoModelForCausalLM.from_pretrained("sdadas/polish-gpt2-small").cuda()

print(len(tokenizer))

for l in langs:
    token_tester(l)

51200
Czech
15678
['Rod', 'i', 'Äį', 'e', 'Ġd', 'ÄĽ', 't', 'ÃŃ', ',', 'Ġkt', 'er', 'Ã©', 'Ġproje', 'v', 'uj', 'ÃŃ', 'Ġz', 'v', 'Ã½', 'Å¡', 'en', 'Ã½', 'Ġz', 'Ã¡', 'jem', 'Ġo', 'Ġn', 'ÄĽ', 'kt', 'er', 'Ã½', 'Ġze', 'Ġsport', 'Å', '¯', ',', 'Ġmaj', 'ÃŃ', 'Ġp', 'ÅĻ', 'ed', 'Ġse', 'bo', 'u', 'Ġne', 'le', 'h', 'k', 'Ã©', 'Ġroz', 'ho', 'do', 'v', 'Ã¡n', 'ÃŃ', '.', 'ĠM', 'ÄĽ', 'li', 'Ġby', 'Ġd', 'ÄĽ', 't', 'ÄĽ', 'm', 'Ġdo', 'v', 'oli', 't', 'Ġtr', 'Ã©', 'no', 'vat', 'Ġproto', ',', 'Ġaby', 'Ġz', 'Ġnich', 'Ġ{', '}', 'ĠÅ', '¡', 'pi', 'Äį', 'kov', 'ÃŃ', 'Ġspor', 'to', 'v', 'ci', 'Ġa', 'Ġspor', 'to', 'v', 'ky', 'n', 'ÄĽ', '?', 'Ġ', 'Ċ'] 100
['Dob', 'r', 'Ã½', 'Ġden', ',', 'Ġj', 'men', 'u', 'ji', 'Ġse', 'ĠKri', 'stina', 'Ġa', 'Ġodpo', 'v', 'ÃŃ', 'd', 'Ã¡', 'm', 'Ġli', 'dem', 'Ġna', 'Ġot', 'Ã¡', 'z', 'ky', ',', 'Ġt', 'Ã½', 'kaj', 'ÃŃ', 'c', 'ÃŃ', 'Ġse', 'Ġjej', 'ich', 'Ġzdra', 'v', 'ÃŃ', '.', 'ĠV', 'Ġtom', 'to', 'Ġro', 'Äį', 'n', 'ÃŃ', 'm', 'Ġob', 'dob', 'ÃŃ', 'Ġdost', 'Ã¡', 'vÃ¡', 'm', 'Ġv', 'Å¾', '

In [68]:
model = BertForMaskedLM.from_pretrained("dkleczek/bert-base-polish-uncased-v1",ignore_mismatched_sizes=True).cuda()
tokenizer = BertTokenizer.from_pretrained("dkleczek/bert-base-polish-uncased-v1")

print(len(tokenizer))

for l in langs:
    token_tester(l)

Some weights of the model checkpoint at dkleczek/bert-base-polish-uncased-v1 were not used when initializing BertForMaskedLM: ['cls.seq_relationship.weight', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


60000
Czech
10234
['rod', '##ice', 'det', '##i', ',', 'kt', '##ere', 'proje', '##vu', '##ji', 'z', '##vy', '##sen', '##y', 'zaje', '##m', 'o', 'nek', '##tery', 'ze', 'sportu', ',', 'maj', '##i', 'pre', '##d', 'se', '##bo', '##u', 'nel', '##eh', '##ke', 'roz', '##hod', '##ov', '##ani', '.', 'meli', 'by', 'det', '##em', 'do', '##vol', '##it', 'tren', '##ov', '##at', 'proto', ',', 'aby', 'z', 'nich', '{', '}', 'spi', '##cko', '##vi', 'sport', '##ov', '##ci', 'a', 'sport', '##ov', '##ky', '##ne', '?'] 66
['dobry', 'den', ',', 'j', '##men', '##uj', '##i', 'se', 'kristi', '##na', 'a', 'odpo', '##vi', '##dam', 'li', '##dem', 'na', 'ota', '##z', '##ky', ',', 'tyka', '##ji', '##ci', 'se', 'jej', '##ich', 'zdra', '##vi', '.', 'v', 'tom', '##to', 'ro', '##c', '##nim', 'ob', '##dob', '##i', 'dosta', '##va', '##m', 'v', '##zdy', 'mno', '##ho', '{', '}', '.'] 49
['zima', 'je', 'ne', '##bez', '##pec', '##na', ',', 'proto', '##ze', 'je', 'tez', '##ke', 've', '##det', ',', 'co', 'se', 'stan', '##e', '.

In [54]:
tokenizer = AutoTokenizer.from_pretrained("sdadas/polish-gpt2-small")
model = AutoModelForCausalLM.from_pretrained("sdadas/polish-gpt2-small").cuda()

for l in langs:
    df = run_tests(model, tokenizer, l)
    df.to_csv("Outputs/GPT_P_Transliterated_"+l+".csv")

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

...

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

In [58]:
model = BertForMaskedLM.from_pretrained("dkleczek/bert-base-polish-uncased-v1",ignore_mismatched_sizes=True).cuda()
tokenizer = BertTokenizer.from_pretrained("dkleczek/bert-base-polish-uncased-v1")

for l in langs:
    df = run_tests(model, tokenizer, l, BERT = True)
    df.to_csv("Outputs/BERT_P_Transliterated_"+l+".csv")

Some weights of the model checkpoint at dkleczek/bert-base-polish-uncased-v1 were not used when initializing BertForMaskedLM: ['cls.seq_relationship.weight', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

...

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]